# 📡 Telecom Analytics - Predicción de Churn
Proyecto desarrollado en Python con scikit-learn.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


## 1️⃣ Carga de datos

In [ ]:

df = pd.read_csv("telecom_analytics_new.csv")
df.head()


## 2️⃣ Preparación de datos

In [ ]:

df["churn"] = df["churn"].map({"Yes":1, "No":0})
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3️⃣ Modelo - Logistic Regression

In [ ]:

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_model.predict_proba(X_test_scaled)[:,1]))


## 4️⃣ Modelo - Random Forest + GridSearch

In [ ]:

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor ROC-AUC:", grid.best_score_)


## 5️⃣ Conclusiones

- Clientes con contrato mensual y baja antigüedad presentan mayor churn.
- Alto número de llamadas a soporte aumenta riesgo.
- Se recomienda estrategia de retención temprana.